# 🧬 EasyPasal: Experiment E5 — DSPy + GEPA Prompt Optimization (T5.1)

This notebook implements the end-to-end **Genetic-Pareto (GEPA) Reflective Prompt Optimization** (*Agrawal et al., 2025*) for Indonesian financial regulatory question-answering (**EasyPasal**).

### Methodology & Protocol:
- **Dataset Partitioning**: `data/gold/dev.jsonl` (N=40) is partitioned into `e5_train` (N=23, 60%) for reflective mutation and `e5_val` (N=17, 40%) for Pareto frontier candidate tracking.
- **Held-Out Generalization**: `data/gold/test.jsonl` (N=66) is kept strictly isolated until the final benchmark to audit against prompt overfitting.
- **Multi-Objective Optimization**: Evaluates candidates across Citation F1, Grounding Fidelity (1 - hallucination rate), Abstention Accuracy, and Lexical F1.
- **Statistical Rigor**: Reports 95% bootstrap confidence intervals (B=2,000) and paired Monte Carlo permutation tests (P=5,000).

In [ ]:
# 1. Environment & Hardware Verification
import sys
from pathlib import Path

print(f"Python Version: {sys.version}")
!nvidia-smi 2>/dev/null || echo "Running in CPU mode"

In [ ]:
# 2. Install Dependencies
!pip install -q gepa==0.1.4 fastembed qdrant-client rank-bm25 llama-cpp-python pydantic tqdm numpy

In [ ]:
# 3. Ensure Repository Setup
import sys

if not Path("src").exists():
    !git clone https://github.com/robitalhazmi/easypasal.git
    %cd easypasal
if "." not in sys.path:
    sys.path.insert(0, ".")
print(f"Repository ready at: {Path.cwd()}")

In [ ]:
# 4. Stratified Dataset Partitioning
from experiments.e5_gepa.dataset import partition_and_save_e5_splits

train_path, val_path = partition_and_save_e5_splits(
    dev_path="data/gold/dev.jsonl",
    output_dir="data/gold",
    train_ratio=0.6,
    seed=42,
)

In [ ]:
# 5. Execute GEPA Evolutionary Optimization Runner
!PYTHONUNBUFFERED=1 python experiments/e5_gepa/run_experiment.py \
    --provider llamacpp \
    --max-metric-calls 12 \
    --output-json results/e5_gepa_results.json

In [ ]:
# 6. Inspect Winning Prompt & Overfitting Audit
import json

with open("results/e5_gepa_results.json", encoding="utf-8") as f:
    res = json.load(f)
print("=" * 70)
print("Winning GEPA Prompt:")
print("=" * 70)
print(res["winning_prompt"])
print("\n" + "=" * 70)
print("Overfitting Audit Summary:")
print("=" * 70)
for k, v in res["overfitting_audit"].items():
    print(f"  • {k}: {v}")

In [ ]:
# 7. Interactive Q&A Demo with GEPA Winning Prompt
from easypasal.generation.pipeline import RAGPipeline

pipeline = RAGPipeline.from_config(config_path="configs/default.yaml")
test_query = "Berapa hari PUJK wajib menyelesaikan pengaduan nasabah?"
out = pipeline.ask(question=test_query, prompt_variant="gepa", verify_citations=True)
print(f"Question : {test_query}")
print(f"Answer   : {out.answer}")
print(f"Citations: {out.citations}")
print(f"Abstained: {out.abstained}")